# Lab 7 — اجعل Bayan أسرع وشغّله كخدمة
**SDA-AIE-211 **

## الهدف
سنقوم بأربع خطوات فقط:
1. نقيس سرعة نموذج التصنيف.
2. نحوله إلى ONNX ثم INT8 ونقارن السرعة والنتائج.
3. نشغّل النموذج داخل API.
4. نختبر الخدمة ونوثّق النتائج.

**قبل البدء:** أكمل Lab 3a، وتأكد من وجود:
- `bayan_topic_model`
- `data/bench_mix.npy`

> شغّل الخلايا بالترتيب. لا تكتب أن الهدف تحقق إلا إذا أظهرت نتائج تشغيلك ذلك.

## 1. تثبيت المكتبات وتحميل النموذج

In [25]:
import zipfile

zip_path = '/content/bayan_topic_model-20261110T102622Z-1-001.zip'

if zipfile.is_zipfile(zip_path):
    with zipfile.ZipFile(zip_path) as z:
        z.extractall('/content')
    print('تم فك الضغط ✅')
else:
    print('ملف ZIP غير صالح؛ أعد تنزيله ورفعه بعد اكتمال التنزيل')

تم فك الضغط ✅


In [27]:
# تثبيت المكتبات المطلوبة (يكفي تشغيله مرة واحدة، ويمكنك وضع # قبله بعد ذلك)
# optimum حُذفت لأنها تتعارض مع إصدار transformers الحديث في Colab؛ نصدّر ONNX بـ torch مباشرة
%pip -q install transformers onnx onnxscript onnxruntime fastapi uvicorn httpx torch numpy

# المكتبات: نكتب فائدة كل مكتبة بجانب اسمها
from pathlib import Path                         # التعامل مع مسارات الملفات
import time                                      # قياس زمن التنفيذ
import re                                        # تنظيف النصوص
import shutil                                    # العثور على أمر تصدير ONNX
import subprocess                                # تشغيل أمر التصدير
import threading                                 # تشغيل API في الخلفية
import numpy as np                               # حساب p50 و p99 ومقارنة النتائج
import torch                                     # تشغيل نموذج PyTorch الأصلي
import onnxruntime as ort                        # تشغيل نموذج ONNX
import httpx                                     # إرسال طلبات HTTP إلى API
import uvicorn                                   # تشغيل خادم FastAPI
from transformers import AutoTokenizer, AutoModelForSequenceClassification  # تحميل tokenizer ونموذج التصنيف
from onnxruntime.quantization import quantize_dynamic, QuantType             # ضغط نموذج ONNX إلى INT8
from fastapi import FastAPI                      # إنشاء API
from pydantic import BaseModel, Field            # التحقق من مدخلات API

# مجلد المشروع على جهازك (غيّره إذا نقلت الملفات)
BASE_DIR = Path("")
print('مجلد العمل | working folder:', BASE_DIR.resolve())
assert BASE_DIR.exists(), f'المجلد غير موجود: {BASE_DIR}'

# البحث عن النموذج وملف الاختبار داخل المجلد وفروعه
def find_first(name, folder_only=False):
    for p in BASE_DIR.rglob(name):
        if p.is_dir() or not folder_only:
            return p
    return None

MODEL_DIR = find_first('bayan_topic_model', folder_only=True)
BENCH_FILE = find_first('bench_mix.npy')
inside = sorted(p.name for p in BASE_DIR.iterdir())   # محتويات المجلد للمساعدة عند الخطأ

assert MODEL_DIR is not None, f'مجلد bayan_topic_model غير موجود. محتويات المجلد: {inside}'
assert (MODEL_DIR / 'config.json').exists(), f'المجلد {MODEL_DIR} لا يحتوي config.json (ليس نموذج transformers)'
assert BENCH_FILE is not None, f'ملف bench_mix.npy غير موجود. محتويات المجلد: {inside}'
DATA_DIR = BENCH_FILE.parent
print('النموذج | model :', MODEL_DIR)
print('الاختبار | bench:', BENCH_FILE)

# تحميل رسائل الاختبار
texts = np.load(BENCH_FILE, allow_pickle=True).tolist()

# تحميل النموذج الأصلي
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).eval()
torch.set_num_threads(4)  # استخدام 4 أنوية CPU

print('عدد رسائل الاختبار:', len(texts))
print('تم تحميل النموذج ✅')

مجلد العمل | working folder: /content
النموذج | model : bayan_topic_model
الاختبار | bench: bench_mix.npy


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

عدد رسائل الاختبار: 2000
تم تحميل النموذج ✅


In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. قياس سرعة النموذج الأصلي

سنقارن إعدادين:
- `FP32 512`: طول 512 مع Padding ثابت.
- `FP32 128`: طول 128 مع Padding حسب طول النص.

**p50** هو الزمن الوسيط، و**p99** يوضح زمن أبطأ 1% من الطلبات.

In [28]:
def predict(text, max_length, fixed_padding):
    # تحويل النص إلى أرقام ثم تشغيل النموذج
    inputs = tokenizer(
        str(text),
        max_length=max_length,
        truncation=True,
        padding='max_length' if fixed_padding else True,
        return_tensors='pt'
    )
    with torch.inference_mode():                 # لا نحتاج حسابات التدريب
        output = model(**inputs)
    return int(output.logits.argmax(-1).item())  # إرجاع رقم الفئة المتوقعة


def benchmark(function):
    times = []
    for i in range(100):                         # نكرر القياس 100 مرة
        start = time.perf_counter()
        function(texts[i % len(texts)])
        times.append((time.perf_counter() - start) * 1000)
    return {
        'p50_ms': round(float(np.percentile(times, 50)), 2),
        'p99_ms': round(float(np.percentile(times, 99)), 2)
    }


# قياس الإعدادين
fp32_512 = benchmark(lambda x: predict(x, 512, True))
fp32_128 = benchmark(lambda x: predict(x, 128, False))

print('FP32 512:', fp32_512)
print('FP32 128:', fp32_128)

FP32 512: {'p50_ms': 1013.21, 'p99_ms': 2261.0}
FP32 128: {'p50_ms': 55.93, 'p99_ms': 186.67}


## 3. تصدير النموذج إلى ONNX ثم INT8

In [ ]:
# إنشاء مجلد لحفظ نموذج ONNX
onnx_dir = BASE_DIR / 'topic_onnx'
onnx_dir.mkdir(exist_ok=True)

# تصدير النموذج إلى ONNX باستخدام torch (بدون Optimum)
class LogitsOnly(torch.nn.Module):               # غلاف بسيط يعيد درجات الفئات (logits) فقط
    def __init__(self, base):
        super().__init__()
        self.base = base
    def forward(self, input_ids, attention_mask):
        return self.base(input_ids=input_ids, attention_mask=attention_mask).logits

export_model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR, attn_implementation='eager').eval()
dummy = tokenizer(['الخدمة لا تعمل'], max_length=128, truncation=True,
                  padding='max_length', return_tensors='pt')      # مثال واحد يحدد شكل المدخلات
onnx_file = onnx_dir / 'model.onnx'
export_args = dict(
    input_names=['input_ids', 'attention_mask'], output_names=['logits'],
    dynamic_axes={'input_ids': {0: 'batch', 1: 'sequence'},       # يقبل أي عدد رسائل وأي طول
                  'attention_mask': {0: 'batch', 1: 'sequence'},
                  'logits': {0: 'batch'}},
    opset_version=17
)
with torch.inference_mode():
    try:
        torch.onnx.export(LogitsOnly(export_model), (dummy['input_ids'], dummy['attention_mask']),
                          str(onnx_file), dynamo=False, **export_args)      # الطريقة المستقرة
    except Exception as exc:
        print('الطريقة الأولى فشلت، نجرب الثانية:', str(exc)[:200])
        torch.onnx.export(LogitsOnly(export_model), (dummy['input_ids'], dummy['attention_mask']),
                          str(onnx_file), dynamo=True, **export_args)       # الطريقة الجديدة (تحتاج onnxscript)
assert onnx_file.exists(), 'لم يُنشأ ملف ONNX'

# إنشاء نسخة INT8 من نموذج ONNX
int8_file = str(BASE_DIR / 'topic_int8.onnx')
quantize_dynamic(str(onnx_file), int8_file, weight_type=QuantType.QInt8) #make int8 copy

# فتح النموذجين باستخدام ONNX Runtime
onnx_model = ort.InferenceSession(str(onnx_file), providers=['CPUExecutionProvider'])
int8_model = ort.InferenceSession(int8_file, providers=['CPUExecutionProvider'])
print('تم إنشاء نسختي ONNX و INT8 ✅')

/var/folders/6g/_9p8xkt10mqgz1f10_wkyl6c0000gn/T/ipykernel_8859/2102220508.py:26: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(LogitsOnly(export_model), (dummy['input_ids'], dummy['attention_mask']),


تم إنشاء نسختي ONNX و INT8 ✅


## 4. قياس السرعة ومقارنة التوقعات

In [ ]:
def predict_onnx(text, session):
    # تجهيز النص بالطريقة نفسها ثم تشغيل نموذج ONNX
    inputs = tokenizer(str(text), max_length=128, truncation=True,
                       padding='max_length', return_tensors='np')
    # نرسل المدخلات التي يحتاجها النموذج فقط
    names = [item.name for item in session.get_inputs()]
    inputs = {name: inputs[name] for name in names}
    output = session.run(None, inputs)[0]
    return int(np.argmax(output, axis=1)[0])


# قياس سرعة ONNX و INT8
onnx_speed = benchmark(lambda x: predict_onnx(x, onnx_model))
int8_speed = benchmark(lambda x: predict_onnx(x, int8_model))
print('ONNX FP32:', onnx_speed)
print('ONNX INT8:', int8_speed)

# مقارنة نتائج INT8 بنتائج النموذج الأصلي على 100 رسالة
sample = texts[:100]
original = [predict(x, 128, False) for x in sample] #original model
compressed = [predict_onnx(x, int8_model) for x in sample] #int8 model
agreement = np.mean(np.array(original) == np.array(compressed)) #إذا كان التوقعان متطابقين نحصل على True (1) حساب نسبة التطابق
print('Prediction agreement:', round(float(agreement), 3))

ONNX FP32: {'p50_ms': 12.98, 'p99_ms': 15.17}
ONNX INT8: {'p50_ms': 10.71, 'p99_ms': 15.36}
Prediction agreement: 0.96


## 5. اختيار النسخة المناسبة

نختار INT8 إذا كانت نتائجه قريبة من النموذج الأصلي. في هذا التمرين نستخدم **97% تطابق** كحد إرشادي، وليس كمعيار عام لكل المشاريع.

نحتفظ دائمًا بنسخة ONNX FP32 كخطة رجوع (Rollback).

In [ ]:
# اختيار النسخة التي ستعمل داخل الخدمة
use_int8 = agreement >= 0.97
selected_model = int8_model if use_int8 else onnx_model

print('النسخة المختارة:', 'INT8' if use_int8 else 'ONNX FP32')
print('سبب الاختيار: مقارنة السرعة مع الحفاظ على تقارب التوقعات.')

النسخة المختارة: ONNX FP32
سبب الاختيار: مقارنة السرعة مع الحفاظ على تقارب التوقعات.


## 6. تجهيز التصنيف وفحوصات البداية (Canary)

الـ Canary اختبار صغير نتأكد به أن تنظيف النص وTokenizer يعملان قبل استقبال المستخدمين.

In [ ]:
# ✍️ TODO 1: استخدم نفس آلية التنظيف التي استخدمتها في Lab 1
# 1) افتح Lab 1 وانسخ الدوال الثلاث (mask_pii و normalize و preprocess) من خلية PREPROCESSING_VERSION.
# 2) الصقها هنا فوق الدالة clean_text مباشرة.
# 3) غيّر جسم الدالة clean_text ليصبح:   return preprocess(text)
# لماذا؟ يجب أن تنظّف الخدمة النص بالطريقة نفسها التي دُرّب بها النموذج، وتُخفي الهاتف والبريد (PII).
def clean_text(text):
    # إزالة التطويل وتوحيد المسافات
    return re.sub(r'\s+', ' ', str(text).replace('ـ', '')).strip()


def classify(text):
    clean = clean_text(text)
    pred_id = predict_onnx(clean, selected_model)
    label = model.config.id2label.get(pred_id, str(pred_id))
    return {'text': clean, 'topic': label, 'model_version': 'bayan-topic-v1'}


# اختبارات بداية بسيطة
assert clean_text('  testـ  text  ') == 'test text'
assert len(tokenizer('الخدمة لا تعمل في الرياض')['input_ids']) > 2
print('Canary checks: GREEN ✅')

# فحص TODO 1: هل أخفت الدالة رقم الهاتف؟
if '[PHONE]' in clean_text('اتصلوا بي على 0551234567'):
    print('TODO 1 ✅ التنظيف مطابق لـ Lab 1')
else:
    print('TODO 1 ⏳ لم تُكمل بعد: رقم الهاتف لا يزال ظاهرًا في النص')
print(classify('تم خصم المبلغ مرتين من الفاتورة'))

Canary checks: GREEN ✅
TODO 1 ✅ التنظيف مطابق لـ Lab 1
{'text': 'تم خصم المبلغ مرتين من الفاتورة', 'topic': 'billing', 'model_version': 'bayan-topic-v1'}


## 7. إنشاء API

Use FastAPI to build one integrated API that connects the Classification, NER, and Semantic Search components to process complaints and return their results.